# Frozen spatial transfer from public external Metrica passes

F1_TARGET.md experiment 17 fixes the model and training protocol before fitting.
Original fold-only preprocessing, five whole-window holdouts with 60-second purge,
three final seeds, fixed 60 epochs. Selection uses training OOF only; only the
selected variant is evaluated on the unchanged, previously inspected validation.
No new manual labels or output corrections. Private match assets stay private.


In [ ]:
from pathlib import Path
import hashlib,json,os,subprocess,sys,torch
assert torch.cuda.is_available()
print('GPU:',torch.cuda.get_device_name(0),flush=True)
INPUT=Path('/kaggle/input');WORK=Path('/kaggle/working/possession_external');WORK.mkdir(parents=True,exist_ok=True)
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
configs=list(INPUT.rglob('external_input.json'));assert len(configs)==1
FILES=configs[0].parent;plan=json.loads(configs[0].read_text())
SAMPLES=FILES/'pretrain_samples.npz';assert sha(SAMPLES)==plan['external_samples_sha256']
roots=[p.parent for p in INPUT.rglob('F1_TARGET.md') if (p.parent/'external_possession/experiment.py').is_file()];assert len(roots)==1
CODE=roots[0];assert sha(CODE/'F1_TARGET.md')==plan['protocol_sha256']
labelplans=list(INPUT.rglob('relational_input.json'));assert len(labelplans)==1
LABELS=labelplans[0].parent
FEATURES=next(p for p in INPUT.rglob('features.npz') if sha(p)==plan['feature_sha256'])
AUDIO=next(p for p in INPUT.rglob('audio_cache.npz') if sha(p)==plan['audio_sha256'])
GRAPH=LABELS/'graph.npz';assert sha(GRAPH)==plan['graph_sha256']
for split in ('train','validation'):assert sha(LABELS/f'{split}.csv')==plan[split+'_sha256']
sources=[p.parent for p in INPUT.rglob('experiment.json') if json.loads(p.read_text()).get('kind')=='team_symmetric_graph_memory_span_ablation'];assert len(sources)==1
SOURCE=sources[0]
controls=[p.parent for p in INPUT.rglob('experiment.json') if json.loads(p.read_text()).get('kind')=='symmetric_factorized_boosted_trees'];assert len(controls)==1
BOOSTED=controls[0]
for name,digest in plan['boosted_control_sha256'].items():assert sha(BOOSTED/name)==digest
DEPS=Path('/tmp/barca_coarse_python')
subprocess.run([sys.executable,'-m','pip','install','--quiet','--disable-pip-version-check','--target',str(DEPS),'-r',str(CODE/'requirements-temporal.txt')],check=True)
subprocess.run([sys.executable,'-m','pip','install','--quiet','--disable-pip-version-check','--target',str(DEPS),'--no-deps','-r',str(CODE/'requirements-foundation.txt')],check=True)
env=dict(os.environ,PYTHONPATH=str(DEPS)+os.pathsep+str(CODE),PYTHONDONTWRITEBYTECODE='1',HF_HUB_OFFLINE='1',CUBLAS_WORKSPACE_CONFIG=':4096:8')
def run(*args):subprocess.run([sys.executable,*map(str,args)],cwd=CODE,env=env,check=True)
run('-m','unittest','discover','-s','tests','-v')
print('Full coarse suite passed; starting fixed real study.',flush=True)


In [ ]:
PRETRAINED=WORK/'pretrained'
run('-m','external_possession.pretrain','--samples',SAMPLES,'--out',PRETRAINED,'--device','cuda')
RUN=WORK/'study'
run('-m','external_possession','compare','--features',FEATURES,'--audio',AUDIO,'--graph',GRAPH,
    '--source-study',SOURCE,'--boosted-study',BOOSTED,'--pretrained',PRETRAINED/'spatial.pt','--train-labels',LABELS/'train.csv',
    '--validation-labels',LABELS/'validation.csv','--out',RUN,'--epochs','60','--seeds','0','1','2',
    '--device','cuda','--bootstrap','1000')


In [ ]:
output=RUN/'selected_regenerated.csv'
run('-m','external_possession','predict','--features',FEATURES,'--audio',AUDIO,'--graph',GRAPH,
    '--model',RUN/'global','--out',output,'--device','cuda')
assert output.read_bytes()==(RUN/'selected_predictions.csv').read_bytes()
source={str(p.relative_to(CODE)):sha(p) for root in ('external_possession','relational_possession','temporal_possession','learned_possession','boosted_possession') for p in (CODE/root).glob('*.py')}
protocol=dict(torch=torch.__version__,cuda=torch.version.cuda,gpu=torch.cuda.get_device_name(0),source_sha256=source,
    private_inputs=plan,git_checkpoint=plan['git_checkpoint'],selected_output_regenerated=True,prediction_sha256=sha(output),
    references_unchanged=all(sha(LABELS/f'{split}.csv')==plan[f'{split}_sha256'] for split in ('train','validation')))
assert protocol['references_unchanged']
(WORK/'protocol.json').write_text(json.dumps(protocol,indent=2)+'\n')
print('Byte-identical selected coarse regeneration.',flush=True)
print('COARSE GPU RUN COMPLETE',flush=True)
